# GACC Climatology Builder — 2005–2024

**Endpoint:** `download-nfdr-daily-summary` (confirmed from sample CSV)
**Columns returned:** `ERC · IC · BI · SC · 1HrFM · 10HrFM · 100HrFM · 1000HrFM · KBDI`
**NFDRType:** `O` = observed (used for climo) · `F` = forecast

**Fixes in this version:**
- tqdm output throttled to prevent IOPub message rate error
- Cell 6 uses exact NFDR column names — no `find_col` guessing needed
- Windows-safe install throughout

---
Run top to bottom. **Cell 5 is fully resumable** — stop and restart anytime.

In [12]:
# Windows-safe install — no ! shell commands
import subprocess, sys
for pkg in ['requests', 'pandas', 'numpy', 'openpyxl', 'tqdm']:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '--quiet'],
        capture_output=True, text=True
    )
    print(f'{"✅" if r.returncode == 0 else "❌"} {pkg}')
print('Done.')

✅ requests
✅ pandas
✅ numpy
✅ openpyxl
✅ tqdm
Done.


In [13]:
import pandas as pd, numpy as np, json, importlib.util, time, requests
from pathlib  import Path
from datetime import date
from io       import StringIO

# ─── Paste your FEMS credentials here ────────────────────────────────────────
FEMS_API_KEY  = '92DQWMKBLAQML2ERV4MD3CPSA6DP3MUM0G5K2I85DZMMLJHFUKPLGD4DGO9U13A8'
FEMS_USERNAME = 'kristen.allison@usda.gov'
# ─────────────────────────────────────────────────────────────────────────────

assert FEMS_API_KEY,  '❌ Add FEMS_API_KEY'
assert FEMS_USERNAME, '❌ Add FEMS_USERNAME'

CLIMO_START = 2005
CLIMO_END   = 2024
PERCENTILES = [80, 90, 95, 97]
RAW_DIR     = Path('fems_raw')
RAW_DIR.mkdir(exist_ok=True)

# Confirmed working endpoint from your sample URL
FEMS_BASE = 'https://fems.fs2c.usda.gov/api/ext-climatology/download-nfdr-daily-summary/'

HEADERS = {
    'x-api-key':    FEMS_API_KEY,
    'x-user-email': FEMS_USERNAME,
    'Accept':       'text/csv',
}

# ── Load GACC config ──────────────────────────────────────────────────────────
spec = importlib.util.spec_from_file_location('gacc_config', 'gacc_config.py')
gc   = importlib.util.module_from_spec(spec)
spec.loader.exec_module(gc)
GACC_CONFIG = gc.GACC_CONFIG

ACTIVE = {
    name: data for name, data in GACC_CONFIG.items()
    if any(p['stations'] for p in data['psas'].values())
}

# station_id → fuel model lookup
station_fuel = {}
for g in ACTIVE.values():
    for info in g['psas'].values():
        fm = info['fuel_model']
        for s in info['stations']:
            station_fuel[s] = fm

all_stations = sorted(station_fuel.keys())
years        = list(range(CLIMO_START, CLIMO_END + 1))
total_jobs   = len(all_stations) * len(years)

print(f'Active GACCs    : {[d["abbrev"] for d in ACTIVE.values()]}')
print(f'Total PSAs      : {sum(len(g["psas"]) for g in ACTIVE.values())}')
print(f'Unique stations : {len(all_stations)}')
print(f'Years           : {CLIMO_START}–{CLIMO_END}  ({len(years)} years)')
print(f'Total requests  : {total_jobs:,}  (~{total_jobs // 2 // 60} min at 0.5s/req)')
print(f'Save folder     : {RAW_DIR.resolve()}')

Active GACCs    : ['EACC', 'GBCC', 'ONCC', 'NRCC', 'NWCC', 'RMCC', 'SACC', 'OSCC', 'SWCC']
Total PSAs      : 213
Unique stations : 874
Years           : 2005–2024  (20 years)
Total requests  : 17,480  (~145 min at 0.5s/req)
Save folder     : C:\Users\kristenallison\fems_raw


In [8]:
# Test the exact endpoint from your confirmed working URL:
# https://fems.fs2c.usda.gov/api/ext-climatology/download-nfdr-daily-summary/
#   ?dataset=all&startDate=2026-02-03&endDate=2026-02-25
#   &dataFormat=csv&stationIds=45438&fuelModels=Y

test_sid  = all_stations[0]
test_fuel = station_fuel[test_sid]

test_url = (
    f'{FEMS_BASE}'
    f'?dataset=all'
    f'&startDate=2015-01-01'
    f'&endDate=2015-12-31'
    f'&dataFormat=csv'
    f'&stationIds={test_sid}'
    f'&fuelModels={test_fuel}'
)

print(f'Station : {test_sid}  (fuel model {test_fuel})')
print(f'URL     : {test_url}\n')

resp = requests.get(test_url, headers=HEADERS, timeout=45)
print(f'HTTP status   : {resp.status_code}')
print(f'Response size : {len(resp.content):,} bytes')

if resp.status_code == 200 and len(resp.text) > 50:
    test_df = pd.read_csv(StringIO(resp.text))
    print(f'\n✅ {len(test_df)} rows')
    print(f'Columns: {list(test_df.columns)}')
    obs = test_df[test_df['NFDRType'] == 'O']
    fct = test_df[test_df['NFDRType'] == 'F']
    print(f'NFDRType O (observed): {len(obs)} rows')
    print(f'NFDRType F (forecast): {len(fct)} rows')
    print()
    print(test_df[['StationName','ObservationTime','NFDRType',
                   'ERC','IC','BI','SC','100HrFM','KBDI']].head(5).to_string())
else:
    print('\n❌ Unexpected response:')
    print(resp.text[:400])

Station : 10402  (fuel model Y)
URL     : https://fems.fs2c.usda.gov/api/ext-climatology/download-nfdr-daily-summary/?dataset=all&startDate=2015-01-01&endDate=2015-12-31&dataFormat=csv&stationIds=10402&fuelModels=Y

HTTP status   : 200
Response size : 373 bytes

✅ 1 rows
Columns: ['StationName', 'ObservationTime', 'NFDRType', 'FuelModel', '1HrFM', 'Min1HrFMTime', '10HrFM', 'Min10HrFMTime', '100HrFM', 'Min100HrFMTime', '1000HrFM', 'Min1000HrFMTime', 'KBDI', 'GSI', 'WoodyFM', 'HerbFM', 'IC', 'MaxICTime', 'ERC', 'MaxERCTime', 'SC', 'MaxSCTime', 'BI', 'MaxBITime', 'NFDRQAFlag']
NFDRType O (observed): 0 rows
NFDRType F (forecast): 0 rows

  StationName ObservationTime NFDRType ERC IC BI SC 100HrFM KBDI
0                                                               


In [14]:
# Confirm all required columns exist in the response
# These are the exact column names from your sample CSV:
#   nfdrsDailySummary2026-02-04_2026-02-25.csv

REQUIRED_COLS = {
    'date':   'ObservationTime',
    'type':   'NFDRType',       # O = observed, F = forecast
    'erc':    'ERC',
    'ic':     'IC',
    'bi':     'BI',
    'sc':     'SC',
    'fm1':    '1HrFM',
    'fm10':   '10HrFM',
    'fm100':  '100HrFM',
    'fm1000': '1000HrFM',
    'kbdi':   'KBDI',
}

print('Column check against test_df from Cell 3:')
all_ok = True
for field_key, col_name in REQUIRED_COLS.items():
    present = col_name in test_df.columns
    status  = '✅' if present else '❌  ← MISSING'
    print(f'  {field_key:8s} → {col_name:20s} {status}')
    if not present:
        all_ok = False

print()
if all_ok:
    print('✅ All columns present — proceed to Cell 5.')
else:
    print('❌ Missing columns detected.')
    print('   Actual columns in response:')
    print('  ', list(test_df.columns))
    print('   Update REQUIRED_COLS above to match.')

Column check against test_df from Cell 3:
  date     → ObservationTime      ✅
  type     → NFDRType             ✅
  erc      → ERC                  ✅
  ic       → IC                   ✅
  bi       → BI                   ✅
  sc       → SC                   ✅
  fm1      → 1HrFM                ✅
  fm10     → 10HrFM               ✅
  fm100    → 100HrFM              ✅
  fm1000   → 1000HrFM             ✅
  kbdi     → KBDI                 ✅

✅ All columns present — proceed to Cell 5.


In [10]:
# Bulk download — resumable, tqdm throttled to avoid IOPub message rate error
#
# The IOPub error you saw happens because tqdm sends a notebook output message
# every update. With 13,984 files at 0.5s each, that's ~7,000 msgs/min — way
# over the 1,000 msg/sec limit. Fix: print progress every 100 files instead,
# and use tqdm with mininterval=10 (update bar at most once every 10 seconds).

from tqdm.notebook import tqdm as tqdm_nb

def download_station_year(station_id, year):
    """
    Download NFDR daily summary for one station / one year.
    Uses the correct fuel model for that station.
    Returns: 'cached' | 'ok' | None (no data for that station-year)
    """
    out_file = RAW_DIR / f'{station_id}_{year}.csv'
    if out_file.exists():
        return 'cached'

    fuel = station_fuel.get(station_id, 'Y')
    url  = (
        f'{FEMS_BASE}'
        f'?dataset=all'
        f'&startDate={year}-01-01'
        f'&endDate={year}-12-31'
        f'&dataFormat=csv'
        f'&stationIds={station_id}'
        f'&fuelModels={fuel}'
    )

    for attempt in range(1, 4):
        try:
            r = requests.get(url, headers=HEADERS, timeout=45)
            if r.status_code == 404:
                out_file.write_text('', encoding='utf-8')
                return None
            r.raise_for_status()
            if not r.text.strip() or len(r.text) < 50:
                out_file.write_text('', encoding='utf-8')
                return None
            out_file.write_text(r.text, encoding='utf-8')
            return 'ok'
        except requests.exceptions.Timeout:
            if attempt < 3: time.sleep(3 * attempt)
        except Exception:
            if attempt < 3: time.sleep(3 * attempt)
            else: return None
    return None


jobs  = [(s, y) for s in all_stations for y in years]
done  = sum(1 for s, y in jobs if (RAW_DIR / f'{s}_{y}.csv').exists())
pend  = len(jobs) - done

print(f'Total   : {len(jobs):,}')
print(f'Cached  : {done:,}')
print(f'Pending : {pend:,}  (~{max(0,pend) // 2 // 60} min)')
print()
print('Progress printed every 200 files to avoid IOPub rate limit.')
print('The tqdm bar updates every 30 seconds.')
print()

new_dl = 0
empty  = 0
errors = []

# mininterval=30  → tqdm only redraws the bar every 30 seconds max
# This keeps notebook output messages well below the 1000/sec IOPub limit
pbar = tqdm_nb(total=len(jobs), initial=done,
               desc='Downloading', unit='file', mininterval=30)

for i, (sid, yr) in enumerate(jobs):
    # Skip already-done files without touching the API
    if (RAW_DIR / f'{sid}_{yr}.csv').exists():
        continue

    result = download_station_year(sid, yr)
    if result == 'ok':
        new_dl += 1
    elif result is None:
        empty += 1
        errors.append((sid, yr))

    pbar.update(1)

    # Plain text progress every 200 new downloads (not every file)
    # — plain print() does NOT count against IOPub display rate
    if new_dl > 0 and new_dl % 200 == 0:
        total_done = done + new_dl + empty
        pct = total_done / len(jobs) * 100
        print(f'  [{pct:5.1f}%]  {total_done:,}/{len(jobs):,}  '
              f'new={new_dl}  no-data={empty}')

    time.sleep(0.5)

pbar.close()
print(f'\n✅ Download complete')
print(f'   New files    : {new_dl:,}')
print(f'   No data      : {empty:,}  (normal — station may not have existed that year)')
if errors[:3]:
    print(f'   Examples     : {errors[:3]}')

Total   : 17,480
Cached  : 17,480
Pending : 0  (~0 min)

Progress printed every 200 files to avoid IOPub rate limit.
The tqdm bar updates every 30 seconds.



Downloading: 100%|##########| 17480/17480 [00:00<?, ?file/s]


✅ Download complete
   New files    : 0
   No data      : 0  (normal — station may not have existed that year)


In [15]:
# Parse all downloaded CSVs and compute PSA-level percentiles
# Uses OBSERVED rows only (NFDRType == 'O') for the climo baseline
#
# Column names are exact — no find_col guessing needed.
# Confirmed from: nfdrsDailySummary2026-02-04_2026-02-25.csv

from tqdm.notebook import tqdm as tqdm_nb

# field_key → exact CSV column name from the NFDR daily summary endpoint
FIELD_COLS = {
    'erc':    'ERC',
    'ic':     'IC',
    'bi':     'BI',
    'sc':     'SC',
    'fm1':    '1HrFM',
    'fm10':   '10HrFM',
    'fm100':  '100HrFM',
    'fm1000': '1000HrFM',
    'kbdi':   'KBDI',
}


def load_station(sid):
    """Load all climo years for one station — observed rows only."""
    frames = []
    for yr in years:
        f = RAW_DIR / f'{sid}_{yr}.csv'
        if not f.exists() or f.stat().st_size < 50:
            continue
        try:
            df = pd.read_csv(f, encoding='utf-8', on_bad_lines='skip')
            df.columns = df.columns.str.strip()

            # Keep observed rows only
            if 'NFDRType' in df.columns:
                df = df[df['NFDRType'] == 'O'].copy()
            if df.empty or 'ObservationTime' not in df.columns:
                continue

            row = pd.DataFrame({
                'date': pd.to_datetime(df['ObservationTime'], errors='coerce')
            })
            for fk, col in FIELD_COLS.items():
                if col in df.columns:
                    row[fk] = pd.to_numeric(df[col].values, errors='coerce')
                else:
                    row[fk] = np.nan

            frames.append(row.dropna(subset=['date']))
        except Exception:
            continue

    if not frames:
        return pd.DataFrame(columns=['date'] + list(FIELD_COLS))
    return pd.concat(frames, ignore_index=True)


# ── Load all stations ─────────────────────────────────────────────────────────
print('Parsing CSV files (observed rows only)...')
station_map = {}
# mininterval=10 keeps tqdm output low
for sid in tqdm_nb(all_stations, desc='Loading', unit='station', mininterval=10):
    df = load_station(sid)
    if not df.empty:
        station_map[sid] = df

print(f'\nStations with usable data: {len(station_map)}/{len(all_stations)}')


# ── Compute PSA-level percentiles ─────────────────────────────────────────────
print('\nComputing PSA-level percentiles...')
result_rows = []

for gacc_name, data in ACTIVE.items():
    for psa_id, info in data['psas'].items():
        sids   = info['stations']
        frames = [station_map[s] for s in sids if s in station_map]
        if not frames:
            continue

        # Average all stations per day, then percentile over all 2005–2020 days
        combined = pd.concat(frames, ignore_index=True)
        daily    = combined.groupby('date').mean(numeric_only=True).dropna(how='all')
        if daily.empty:
            continue

        row = {
            'gacc':        gacc_name,
            'abbrev':      data['abbrev'],
            'psa':         psa_id,
            'fuel_model':  info['fuel_model'],
            'n_stations':  len(sids),
            'n_with_data': len(frames),
            'n_days':      len(daily),
        }
        for fk in FIELD_COLS:
            s = daily[fk].dropna() if fk in daily.columns else pd.Series(dtype=float)
            if s.empty:
                continue
            row[f'{fk}_mean'] = round(float(s.mean()), 1)
            for p in PERCENTILES:
                row[f'{fk}_p{p}'] = round(float(np.percentile(s, p)), 1)
        result_rows.append(row)

results = pd.DataFrame(result_rows)
print(f'\n✅ {len(results)} PSAs computed')
show = ['gacc','psa','fuel_model','n_days',
        'erc_mean','erc_p80','erc_p90','erc_p95','erc_p97',
        'bi_mean','bi_p90','fm100_mean','fm100_p90']
print()
print(results[[c for c in show if c in results.columns]].to_string(index=False))

Parsing CSV files (observed rows only)...


Loading:   0%|          | 0/874 [00:00<?, ?station/s]


Stations with usable data: 857/874

Computing PSA-level percentiles...

✅ 209 PSAs computed

                                                   gacc   psa fuel_model  n_days  erc_mean  erc_p80  erc_p90  erc_p95  erc_p97  bi_mean  bi_p90  fm100_mean  fm100_p90
                       Eastern Area Coordination Center  EA01          Z    7305      16.5     26.7     38.3     51.2     57.7     21.7    53.4        19.4       23.2
                       Eastern Area Coordination Center  EA02          Z    7305      19.6     28.5     42.3     55.2     60.4     26.8    56.2        18.7       23.0
                       Eastern Area Coordination Center  EA03          Z    7305      15.2     25.3     36.8     50.4     58.4     19.4    50.1        19.7       23.2
                       Eastern Area Coordination Center  EA04          Z    7305      28.3     52.8     60.4     65.5     68.1     26.9    55.8        19.7       23.2
                       Eastern Area Coordination Center  EA05          

In [16]:
# Save outputs

# 1. JSON — upload this to GitHub
baseline_psa = {}
for _, row in results.iterrows():
    key   = f"{row['gacc']}|{row['psa']}"
    entry = {'fuel_model': row['fuel_model'], 'n_days': int(row['n_days'])}
    for fk in FIELD_COLS:
        mk = f'{fk}_mean'
        if mk in row and pd.notna(row[mk]):
            entry[fk] = {
                'mean': row[mk],
                **{f'p{p}': row.get(f'{fk}_p{p}') for p in PERCENTILES}
            }
    baseline_psa[key] = entry

json_out = {
    'meta': {
        'climo_start': CLIMO_START,
        'climo_end':   CLIMO_END,
        'percentiles': PERCENTILES,
        'fields':      list(FIELD_COLS.keys()),
        'generated':   str(date.today()),
        'n_psa':       len(baseline_psa),
        'endpoint':    'download-nfdr-daily-summary',
        'nfdr_type':   'O (observed only)',
    },
    'psa': baseline_psa
}

Path('gacc_climo_baseline.json').write_text(
    json.dumps(json_out, indent=2), encoding='utf-8'
)
print('✅ gacc_climo_baseline.json  ← upload this to GitHub')

# 2. CSV
results.to_csv('gacc_climo_baseline.csv', index=False, encoding='utf-8')
print('✅ gacc_climo_baseline.csv')

# 3. Excel — one tab per GACC
with pd.ExcelWriter('gacc_climo_baseline.xlsx', engine='openpyxl') as writer:
    for gacc_name in results['gacc'].unique():
        abbrev   = GACC_CONFIG[gacc_name]['abbrev']
        sheet_df = results[results['gacc'] == gacc_name].drop(columns=['gacc'])
        sheet_df.to_excel(writer, sheet_name=abbrev, index=False)
        ws = writer.sheets[abbrev]
        for col in ws.columns:
            ws.column_dimensions[col[0].column_letter].width = 15
print('✅ gacc_climo_baseline.xlsx  (one tab per GACC)')
print()
print('━' * 56)
print('NEXT STEPS')
print('  1. Upload gacc_climo_baseline.json to GitHub')
print('     (same folder as app.py and gacc_config.py)')
print('  2. Redeploy / refresh Streamlit Cloud')
print('  3. Dashboard reads thresholds from JSON —')
print('     FEMS only called for the live 7-day forecast.')
print('━' * 56)

✅ gacc_climo_baseline.json  ← upload this to GitHub
✅ gacc_climo_baseline.csv
✅ gacc_climo_baseline.xlsx  (one tab per GACC)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
NEXT STEPS
  1. Upload gacc_climo_baseline.json to GitHub
     (same folder as app.py and gacc_config.py)
  2. Redeploy / refresh Streamlit Cloud
  3. Dashboard reads thresholds from JSON —
     FEMS only called for the live 7-day forecast.
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
